# 01 · Data pipeline — Stages 1–3
### Predicting impending failures in OpenRCA — **Bank** and **Telecom**

**Research question.** How effectively can LSTM networks predict impending
software-system failures from multivariate telemetry — and do they beat classical
models given the same features?

This notebook prepares the data and stops before any model is fit. Modelling and
evaluation live in **`02_models_evaluation.ipynb`**.

| Stage | Here |
|---|---|
| **1** | Align raw per-day CSVs onto a 1-minute UTC+8 grid, fill gaps, build labels + masks |
| **2** | Split by **day** — chronological and leave-2-days-out CV. Never by row |
| **3** | Window (60-min look-back), robust-scale, select features — all fit on train only |
| 4–6 | → notebook 02: models, evaluation, ablations |

**Two systems, one pipeline.** Set `DATASET` in the next cell to `"bank"` or
`"telecom"` and re-run — no kernel restart, and everything downstream (build folder,
results folder, figures) follows automatically. They are genuinely different systems,
not two copies of the same shape:

| | Bank | Telecom |
|---|---|---|
| coverage | 10 days × 1,440 min | 15 days × **360 min** (00:00–05:59 UTC+8 only) |
| timestamps | seconds | **milliseconds** |
| modalities | container KPIs + app transactions | node / container / middleware / service + app |
| components | 15 | 47 (`os_*`, `docker_*`, `db_*`, `redis_*`) |
| faults | 136, 8 reasons | 51, 5 reasons |

Adding a third system means one entry in `config.DATASETS` — nothing else is
dataset-specific. (Market isn't registered yet: it splits into `cloudbed-1` /
`cloudbed-2` sub-systems and needs one entry per bed.)

### ⚠️ The timezone trap
Every timestamp in OpenRCA — `record.csv`, the metric files, and the `2021_03_04/`
folder names — is **UTC+8**. Render epochs in your local zone and faults land on the
wrong day, on innocent components, and the labels quietly decorrelate from the
telemetry. This pipeline joins and windows **only on raw epoch seconds**, renders time
with `TZ = UTC+8` and nothing else, and derives day membership from the UTC+8 date —
never `epoch // 86400`. Importing `pipeline` asserts that our rendering reproduces
`record.csv`'s own `datetime` column for all 136 faults; it raises if it doesn't.

### Requirements
```
pip install pandas numpy pyarrow matplotlib scikit-learn        # required
pip install xgboost torch                                       # notebook 02
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # so `pipeline` resolves next to the notebook

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from pipeline import *          # config, data, folds, windowing -- see pipeline.py
apply_plot_style()

DATASET = "bank"                  # <-- "bank" or "telecom"; re-run this cell to switch
A = load_into(globals(), DATASET) # injects F, GRID, DAYS, ONSETS, MASKS, BUILD, FIGS, ...

print(summary())
print("\nAll registered datasets:")
print(dataset_overview().to_string())

## Stage 1 — the aligned matrices

`build_dataset.py` does the heavy lifting and caches its output; importing `pipeline`
runs it only if `build/` is missing (~3 min). What it does per day: pivot to
`timestamp × (component||kpi)`, reindex onto the exact 1,440-minute UTC+8 grid, append
the 44 app-metric series, keep only series present on **all ten days**, drop series
over 80 % missing, forward-fill within a day by each series' own cadence, then emit
labels and masks.

| object | contents |
|---|---|
| `F` | 14,400 × 1,565 gap-filled features — **model input** |
| `F_RAW` | the same grid before filling (for diagnostics) |
| `Y_SYS` | 14,400 × 6 system-level labels `h1 … h30` |
| `Y_COMP` | 14,400 × (6 × 14) per-component labels |
| `MASKS` | `in_fault`, `eval_h1 … eval_h30` |
| `ONSETS` | 136 faults, epoch + UTC+8 datetime |

Two decisions that make the numbers honest. **Blackout:** `record.csv` gives an onset
but no duration, so minutes from onset to +10 are excluded from the negative pool —
the model is never rewarded for "predicting" an incident already underway.
**Window completeness:** a row is evaluable only if its 60-minute look-back *and* its
H-minute look-ahead fit inside the same UTC+8 day. The days are not contiguous; a
window straddling 03-12 → 03-23 would be nonsense.

In [ ]:
print((BUILD / "report.md").read_text())

kept = FEATIX[FEATIX["kept_after_fill"]]
per_comp = (kept.groupby("component")
                .agg(series=("feature", "size"), mean_missing_raw=("missing_raw", "mean"))
                .sort_values("series", ascending=False))
per_comp["mean_missing_raw"] = per_comp["mean_missing_raw"].round(3)
print(f"\nRetained series per component (top 20 of {len(per_comp)}):")
print(per_comp.head(20).to_string())

if kept["family"].replace("", np.nan).notna().any():
    print("\nBy modality family:")
    print(kept.groupby("family")["feature"].size().to_string())

gone = sorted(set(FEATIX.loc[~FEATIX["kept_after_fill"], "component"]) - set(per_comp.index))
lost = sorted(set(pd.unique(ONSETS["component"].astype(str))) - set(per_comp.index))
if gone:
    print(f"\nComponents with no surviving series: {gone}")
print("Components that fail but have no telemetry left:", lost or "none")
print("\nAnything dropped here failed the present-on-every-day or >80%-missing rule.")
print("Name it in the write-up rather than letting it vanish -- in Bank that is")
print("dockerA1/A2/B1/B2, which disappear from the telemetry by 03-24.")

## What the data looks like before any modelling

Four things worth knowing before you trust a number later: how the 136 faults are
distributed, how ragged the sampling is, how the positive rate scales with the
horizon, and whether a fault is visible in the telemetry at all.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2))

# (a) faults per day -------------------------------------------------------
ax = axes[0, 0]
per_day = ONSETS["datetime"].str[:10].value_counts().reindex(DAYS, fill_value=0)
ax.bar(range(len(DAYS)), per_day.values, color=PAL[0], width=0.62)
for i, v in enumerate(per_day.values):
    ax.text(i, v + 0.4, str(v), ha="center", fontsize=8, color=MUTED)
ax.set_xticks(range(len(DAYS)))
ax.set_xticklabels([d[5:] for d in DAYS], rotation=45, ha="right")
ax.set_title("Faults per day (UTC+8)")
ax.set_ylabel("faults"); ax.set_ylim(0, per_day.max() * 1.25)
# mark the largest calendar gap between consecutive day folders
_d = pd.to_datetime(pd.Series(DAYS))
_g = _d.diff().dt.days.fillna(1)
if _g.max() > 1:
    i = int(_g.idxmax())
    ax.axvline(i - 0.5, color=MUTED, ls=":", lw=1.2)
    ax.text(i - 0.42, per_day.max() * 1.12, f"{int(_g.max())}-day gap",
            fontsize=8, color=MUTED)

# (b) faults by reason -----------------------------------------------------
ax = axes[0, 1]
by_reason = ONSETS["reason"].value_counts().sort_values()
ax.barh(range(len(by_reason)), by_reason.values, color=PAL[1], height=0.62)
ax.set_yticks(range(len(by_reason)))
ax.set_yticklabels([r if len(r) < 30 else r[:28] + "..." for r in by_reason.index], fontsize=8)
for i, v in enumerate(by_reason.values):
    ax.text(v + 0.45, i, str(v), va="center", fontsize=8, color=MUTED)
ax.set_title("Faults by injected reason"); ax.set_xlabel("faults")
ax.set_xlim(0, by_reason.max() * 1.2); ax.grid(axis="y", visible=False)

# (c) sampling raggedness --------------------------------------------------
ax = axes[1, 0]
bins = np.linspace(0, 1, 41)
ax.hist(F_RAW.isna().mean().values, bins=bins, color=PAL[2], alpha=0.85, label="raw 1-min grid")
ax.hist(F.isna().mean().values, bins=bins, color=PAL[3], alpha=0.85, label="after gap-fill")
ax.set_title("Missingness per series"); ax.set_xlabel("fraction of minutes missing")
ax.set_ylabel("series"); ax.legend(loc="upper right")

# (d) positive rate vs horizon --------------------------------------------
ax = axes[1, 1]
rates = [Y_SYS[f"h{h}"][MASKS[f"eval_h{h}"].astype(bool)].mean() * 100 for h in HORIZONS]
ax.plot(HORIZONS, rates, "-o", color=PAL[0], markersize=6)
for h, r in zip(HORIZONS, rates):
    ax.annotate(f"{r:.1f}%", (h, r), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=8, color=MUTED)
ax.set_title("System-level positive rate by horizon")
ax.set_xlabel("horizon H (minutes)"); ax.set_ylabel("% of evaluable minutes")
ax.set_ylim(0, max(rates) * 1.28)

fig.suptitle(f"OpenRCA {DATASET.title()} — {len(DAYS)} days, {F.shape[0]:,} minutes, "
             f"{len(ONSETS)} fault events", fontsize=12, fontweight="semibold", y=0.995)
fig.tight_layout(); fig.savefig(FIGS / "01_dataset_overview.png", bbox_inches="tight")
plt.show()

In [ ]:
# Is a fault actually visible in the telemetry? Plot the failing component's own
# KPIs around one onset, ranked by how much they move in the 15 minutes before it.
# First fault with >= 30 min of same-day history (Telecom's first starts at 00:05),
# and a window clipped to that day -- never read across a day boundary.
ev = next(e for _, e in ONSETS.sort_values("onset").iterrows()
          if POS_IN_DAY[ROW_OF[int(e["onset"])]] >= 30)
i0 = ROW_OF[int(ev["onset"])]
p0 = int(POS_IN_DAY[i0])
lo, hi = i0 - min(90, p0), i0 + min(60, DAY_MINUTES - p0)
cols = [c for c in FEATS if c.startswith(str(ev["component"]) + "||")]
win  = F.iloc[lo:hi][cols]
pre, base = F.iloc[i0 - 15:i0][cols].mean(), F.iloc[lo:i0 - 15][cols]
shift = ((pre - base.mean()) / base.std().replace(0, np.nan)).abs().sort_values(ascending=False)
show = shift.dropna().head(4).index.tolist()

fig, ax = plt.subplots(figsize=(11.5, 4.0))
mins = (win.index.to_numpy() - int(ev["onset"])) / 60.0
for k, c in enumerate(show):
    v = win[c].to_numpy(dtype=float)
    rng = np.nanmax(v) - np.nanmin(v)
    ax.plot(mins, (v - np.nanmin(v)) / rng if rng > 0 else v * 0,
            color=PAL[k], label=c.split("||")[1][:52])
ax.axvline(0, color=INK, lw=1.4)
ax.set_title(f"{ev['component']} — the 4 KPIs that move most before onset (min-max scaled)\n"
             f"onset {ev['onset_dt_utc8']} UTC+8 · {ev['reason']}", fontsize=11)
ax.set_xlabel("minutes relative to fault onset"); ax.set_ylabel("scaled value")
ax.set_ylim(-0.08, 1.2); ax.legend(loc="upper left", ncol=2)
fig.tight_layout(); fig.savefig(FIGS / "02_fault_closeup.png", bbox_inches="tight")
plt.show()

## Stage 2 — splitting **by day**, never by row

Adjacent minutes are near-duplicates. A random row split leaks the target period into
training and hands you a 0.99 AUC that means nothing. Two protocols:

* **`chrono`** — train on the early days, validate on one, test on the latest ~30 %.
  The deployment-realistic number, and the hard one: in Bank the March 23–25 test block
  has a different component inventory from the training days.
* **`cv`** — leave-2-days-out, so every day is tested once (5 folds for Bank, 8 for
  Telecom). Higher numbers, less realistic.

Report both and explain the gap. The validation day exists for exactly two jobs:
choosing the alarm threshold and early stopping. Nothing is ever selected on test.

In [ ]:
FOLDS = build_folds(["chrono", "cv"])
for f in FOLDS:
    print(f)

role_color = {"train": PAL[0], "val": PAL[4], "test": PAL[1]}
fig, ax = plt.subplots(figsize=(11.5, 0.52 * len(FOLDS) + 1.6))
for r, f in enumerate(FOLDS):
    for c, day in enumerate(DAYS):
        role = "train" if day in f.train else "val" if day in f.val else "test"
        ax.barh(r, 0.86, left=c, height=0.66, color=role_color[role])
ax.set_yticks(range(len(FOLDS)))
ax.set_yticklabels([f"{f.name}  ({f.n_events()} ev)" for f in FOLDS], fontsize=9)
ax.set_xticks(np.arange(len(DAYS)) + 0.43)
ax.set_xticklabels([d[5:] for d in DAYS], rotation=45, ha="right")
ax.invert_yaxis(); ax.grid(visible=False); ax.set_xlim(-0.2, len(DAYS))
handles = [plt.Line2D([], [], color=v, lw=7, label=k) for k, v in role_color.items()]
ax.legend(handles=handles, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.22))
ax.set_title("Day-grouped splits (each row is one fold)", pad=24)
fig.tight_layout(); fig.savefig(FIGS / "03_splits.png", bbox_inches="tight")
plt.show()

## Stage 3 — windowing, scaling, feature selection

Three rules, all of them about not cheating:

1. **Scaler and feature selection are fit on training rows only**, refit inside every
   fold. These KPIs are heavy-tailed, so scaling is median/IQR, not mean/std.
2. **A window never crosses a day boundary** — Stage 1's `eval_h{H}` masks already
   guarantee 60 minutes of history and H minutes of future inside the same UTC+8 day.
3. **Negatives are subsampled for training only.** Test sees every evaluable minute,
   so the reported positive rate is the real one.

Materialising every window at full width would cost ~4.6 GB. Two cuts: select the
top-`k` features on the train fold (ANOVA F), and cap the training window count.
`fold_data(fold, h)` returns everything a model needs for one (fold, horizon).

In [ ]:
H_CHECK = 5 if 5 in HORIZONS else HORIZONS[0]
d = fold_data(FOLDS[0], H_CHECK)

for s in ("train", "val", "test"):
    X, y = d[f"X_{s}"], d[f"y_{s}"]
    print(f"{s:5s} windows {str(X.shape):>20s}  positives {int(y.sum()):5d} "
          f"({y.mean():6.2%})  {X.nbytes / 1e6:6.1f} MB")

print(f"\nTop 10 features selected on the training fold (ANOVA F vs the h={H_CHECK} label):")
for c in d["cols"][:10]:
    print("   ", FEATS[c])

sel_comp = pd.Series(COMPONENT_OF[d["cols"]]).value_counts()
print(f"\nSelected features span {len(sel_comp)} components:")
print(sel_comp.to_string())

In [ ]:
# Leakage guards -- run these every time you change Stage 1 or Stage 3.
tr, va, te = (set(d[f"rows_{s}"].tolist()) for s in ("train", "val", "test"))
assert not (tr & va) and not (tr & te) and not (va & te), "row overlap between splits"
assert {DAY_DATE[r] for r in tr}.isdisjoint({DAY_DATE[r] for r in te}), "day overlap"

for s in ("train", "val", "test"):
    rows = d[f"rows_{s}"]
    starts, ends = rows - LOOKBACK + 1, rows
    assert (POS_IN_DAY[starts] <= POS_IN_DAY[ends]).all(), f"{s}: window crosses a day"
    assert (POS_IN_DAY[rows] >= LOOKBACK - 1).all(), f"{s}: incomplete look-back"
    assert (POS_IN_DAY[rows] < DAY_MINUTES - d["h"]).all(), f"{s}: incomplete look-ahead"

m5 = MASKS[f"eval_h{H_CHECK}"].to_numpy().astype(bool)
expected = FOLDS[0].rows("test"); expected = expected[m5[expected]]
assert np.array_equal(expected, d["rows_test"]), "test set was subsampled -- it must not be"

scaled = d["X_train"]
print("all leakage guards passed")
print(f"scaled window stats: mean {scaled.mean():+.3f}  std {scaled.std():.3f}  "
      f"min {scaled.min():.1f}  max {scaled.max():.1f}  (clipped at ±10)")
print(f"test positive rate {d['y_test'].mean():.3%} == full-data rate for h={H_CHECK}")

## Bank vs Telecom, side by side

Both systems go through the identical Stage 1–3 code, so they can be compared without
caveats about preprocessing. The shapes they land in are quite different, and those
differences are what the modelling results later have to be read against.

In [ ]:
ov = dataset_overview()
print(ov.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))

ax = axes[0]
for k, name in enumerate(ov.index):
    pr = positive_rates(name)
    ax.plot(pr.index, pr.values * 100, "-o", markersize=5,
            color=DATASET_COLOR.get(name, PAL[k]), label=name)
    ax.annotate(name, (pr.index[-1], pr.values[-1] * 100), textcoords="offset points",
                xytext=(6, 0), fontsize=8, color=DATASET_COLOR.get(name, PAL[k]), va="center")
ax.set_title("Positive rate by horizon"); ax.set_xlabel("horizon H (min)")
ax.set_ylabel("% of evaluable minutes"); ax.set_xlim(0, max(HORIZONS) * 1.3)

ax = axes[1]
x = np.arange(len(ov))
ax.bar(x, ov["minutes"], color=[DATASET_COLOR.get(n, PAL[0]) for n in ov.index], width=0.55)
for i, (m, d, mm) in enumerate(zip(ov["minutes"], ov["days"], ov["min_per_day"])):
    ax.text(i, m * 1.02, f"{m:,} min" + chr(10) + f"({d}d x {mm}m)",
            ha="center", fontsize=8, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels(ov.index)
ax.set_title("Labelled minutes available"); ax.set_ylim(0, ov["minutes"].max() * 1.28)
ax.set_ylabel("minutes")

ax = axes[2]
ax.bar(x - 0.17, ov["faults"], width=0.34, color=PAL[0], label="fault events")
ax.bar(x + 0.17, ov["components"], width=0.34, color=PAL[2], label="components")
for i, (f_, c_) in enumerate(zip(ov["faults"], ov["components"])):
    ax.text(i - 0.17, f_ + 2, str(f_), ha="center", fontsize=8, color=MUTED)
    ax.text(i + 0.17, c_ + 2, str(c_), ha="center", fontsize=8, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels(ov.index); ax.legend()
ax.set_title("Events and components"); ax.set_ylim(0, max(ov["faults"].max(), ov["components"].max()) * 1.25)

fig.tight_layout()
fig.savefig(HERE / "results" / "00_dataset_comparison.png", bbox_inches="tight")
plt.show()

print(f"\nactive dataset is still {active().DATASET} -- comparison does not switch it")

### What the comparison means for the modelling

* **Telecom has a quarter of Bank's labelled minutes and a third of its events** (51 vs
  136). Bootstrap CIs will be correspondingly wide — with 51 events, a 0.1 difference in
  event recall is roughly one fault. Do not rank models on Telecom by small margins.
* **The positive rates track each other almost exactly** (~1 % at H=1 to ~30 % at H=30),
  which is a coincidence of injection density, not a shared mechanism. It does make the
  horizon-by-horizon comparison across systems fair.
* **Telecom days are 6 hours long, and they are the *same* 6 hours** (00:00–05:59). A
  "time-of-day" feature therefore means something very different in each system, and the
  `time_of_day` control model in notebook 02 is correspondingly less informative on
  Telecom — it has only 360 distinct clock positions to work with.
* **Telecom spreads 47 components across four modality families** (node, container,
  middleware, service) where Bank has 15 in one. The per-family ablation in notebook 02
  is worth more here.

## Next

Stages 1–3 are done and cached. Open **`02_models_evaluation.ipynb`** — it imports the
same `pipeline` module, so everything defined here is available there without rerunning
the build.

Both datasets are cached in memory, so switching `DATASET` at the top and re-running is
instant after the first load. Stage 1 outputs live in `build/<dataset>/`, results and
figures in `results/<dataset>/`.

Knobs worth turning before you go: `PRESET` (`smoke` / `standard` / `full`, set via the
`OPENRCA_PRESET` environment variable or by editing `pipeline.py`) and, per dataset in
`config.py`, `lookback`, `blackout`, `max_missing`, `min_day_coverage`, `ffill_factor` —
changing any of those means deleting `build/<dataset>/` and rebuilding.